In [3]:
# Install required Groq and OpenAI SDKs quietly
!pip install -q groq openai

import os
from google.colab import userdata
from groq import Groq

# Load Groq API key from Colab Secrets and initialize the client
os.environ["GROQ_API_KEY"] = userdata.get("Groq")
client = Groq()
MODEL = "openai/gpt-oss-20b"

In [4]:
import json, functools
from openai import RateLimitError, APIConnectionError

# Store menu items and their prices in rupees
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

# Build system prompt by injecting MENU as JSON along with rules and format
SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

In [5]:
# Decorator to log every function call and its return value
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

# Calculate total item price while automatically logging the call
@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

# Initialize conversation memory with the system prompt at index 0
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    # Append user message and keep only system prompt + last MAX_TURNS messages
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        # Call Groq API with trimmed conversation history
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    # Save assistant reply to history so memory persists in the next turn
    history.append({"role": "assistant", "content": reply})
    return reply



In [6]:
# Test the decorated price function and start the interactive chat loop
print("Total for 6 cupcakes:", get_price("cupcake", 6))
while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

   [log] get_price('cupcake', 6) -> 480
Total for 6 cupcakes: 480
You: quit
Bot: Thank you for visiting Sweet Crumbs!


In [7]:
# Create a version of the system prompt with the RULES section removed
SYSTEM_NO_RULES = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

# Compare how the bot responds to a 1-hour delivery request without vs. with RULES
for label, sys_prompt in [("WITHOUT RULES", SYSTEM_NO_RULES), ("WITH RULES", SYSTEM)]:
    res = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": sys_prompt}, {"role": "user", "content": "Can you deliver in 1 hour?"}],
        temperature=0.3
    ).choices[0].message.content
    print(f"{label}: {res}\n")

WITHOUT RULES: Hi! Thanks for reaching out. We need a 24‑hour notice for orders, so we can’t deliver in 1 hour. All cakes are eggless. Let us know if you’d like to place an order for a later date. Cheers!

WITH RULES: I’m not sure about a 1‑hour delivery. I’ll check with the owner and get back to you shortly. Thanks for reaching out!



In [8]:
# Define 3 labeled examples to teach the model strict message classification
CLASSIFIER_PROMPT = """Classify the customer message into ORDER, PRICE_INQUIRY, or DELIVERY. Reply with only the label.

Customer: "I want 2 plum cakes for tomorrow." -> ORDER
Customer: "How much is a chocolate cake?" -> PRICE_INQUIRY
Customer: "Can you deliver to Kakinada?" -> DELIVERY"""

tests = [
    "What is the cost of 6 cupcakes?",
    "Book 1 eggless chocolate cake for Friday.",
    "Do you offer home delivery?",
    "Price for plum cake please?",
    "I'd like to order 3 cupcakes."
]

# Run the classifier with temperature=0 on 5 new customer messages
for msg in tests:
    label = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": CLASSIFIER_PROMPT}, {"role": "user", "content": f'Customer: "{msg}" ->'}],
        temperature=0
    ).choices[0].message.content.strip()
    print(f"{msg} => {label}")

What is the cost of 6 cupcakes? => PRICE_INQUIRY
Book 1 eggless chocolate cake for Friday. => ORDER
Do you offer home delivery? => DELIVERY
Price for plum cake please? => PRICE_INQUIRY
I'd like to order 3 cupcakes. => ORDER


In [9]:
# Simulate an order conversation and request the summary strictly in JSON format
msgs = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Hi, I'm Ravi. I want 2 eggless chocolate cakes for tomorrow."},
    {"role": "assistant", "content": "Noted Ravi, 2 eggless chocolate cakes for tomorrow (Rs 1,300)."},
    {"role": "user", "content": "Return the order as JSON with keys: name, item, qty, date, eggless. No extra text."}
]

raw = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0).choices[0].message.content
print("Raw:", raw)

# Safely parse the returned JSON string into a Python dictionary
try:
    order = json.loads(raw.strip().strip("`").removeprefix("json").strip())
    print("Parsed:", order)
except json.JSONDecodeError as e:
    print("JSON Error:", e)

Raw: {"name":"Ravi","item":"chocolate cake","qty":2,"date":"tomorrow","eggless":true}
Parsed: {'name': 'Ravi', 'item': 'chocolate cake', 'qty': 2, 'date': 'tomorrow', 'eggless': True}


In [10]:
# Shrink memory window to 2 messages and reset conversation history
MAX_TURNS = 2
history = [{"role": "system", "content": SYSTEM}]

# Introduce name, chat for 3 turns, then ask for the name again
for msg in ["My name is Ravi.", "What cakes do you have?", "Are they eggless?", "How much notice?", "What is my name?"]:
    print(f"You: {msg}\nBot: {chat(msg)}\n")

# Explanation: MAX_TURNS = 2 keeps only the last 2 messages (1 user + 1 bot turn) after the system prompt,
# so after 3 turns, "My name is Ravi." gets sliced out of history and the bot forgets it.

You: My name is Ravi.
Bot: Hi Ravi! 👋 I’ve noted your name. How can I help you today?

You: What cakes do you have?
Bot: Hi Ravi! 👋 We offer plum cake ₹500, chocolate cake ₹650, and cupcakes ₹80. All are egg‑free. Need any of these?

You: Are they eggless?
Bot: Yes, all our cakes—including plum cake, chocolate cake, and cupcakes—are available in eggless versions. Let me know which one you’d like!

You: How much notice?
Bot: We need 24 hours’ notice before placing an order.

You: What is my name?
Bot: I don’t have your name yet. Could you please tell me?



In [11]:
# Define service catalogue and system prompt for ShareSphere
CATALOGUE = {"web dev": 800, "ui/ux": 500, "ppt": 250, "data analytics": 700}

BIZ_SYSTEM = f"""ROLE:   You are the friendly assistant for ShareSphere, a student freelancing platform.
FACTS:  Services in rupees: {json.dumps(CATALOGUE)}
        Only verified college students can register. Payouts via UPI within 24 hours.
RULES:  Never share student phone numbers. Never promise exact delivery times.
        If it isn't in FACTS, say "I'll check with the team." Remember name and service.
FORMAT: Under 50 words, friendly, English or Telugu matching the student."""

# Decorated function with docstring to calculate total service cost
@log_call
def get_service_price(service: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of a service in the catalogue."""
    return CATALOGUE[service.lower().strip()] * qty

# Reset history with ShareSphere's system prompt and restore MAX_TURNS = 8
history = [{"role": "system", "content": BIZ_SYSTEM}]
MAX_TURNS = 8

# Test pricing function and run 3 sample customer turns
print("Total for 2 PPTs:", get_service_price("ppt", 2))
for q in ["Hi, I am Sharif. I need 2 ui/ux tasks.", "Can you give me the designer's phone number?", "What is my name and order?"]:
    print(f"You: {q}\nBot: {chat(q)}\n")

   [log] get_service_price('ppt', 2) -> 500
Total for 2 PPTs: 500
You: Hi, I am Sharif. I need 2 ui/ux tasks.
Bot: Hi Sharif! Each UI/UX task is ₹500. Two tasks would be ₹1,000. Let me know if you’d like to proceed and we’ll set it up.

You: Can you give me the designer's phone number?
Bot: I’m sorry, but I can’t share phone numbers. I can help you connect with a designer through the platform instead.

You: What is my name and order?
Bot: Your name is Sharif, and you’ve ordered 2 UI/UX tasks.

